# Bài 6 · Kết nối & truy xuất dữ liệu ngoài

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Viện TTNT, UET-VNU**

> 💡 File → **Save a copy in Drive** trước khi sửa.

**Mục tiêu buổi học** — sau notebook này, bạn sẽ:

1. Đọc file nén, nhiều cột bằng `read_csv` (chọn cột, khai báo kiểu, parse ngày) và dùng **Parquet**
   cho dữ liệu trung gian.
2. Gọi một API thật bằng `requests` (params, timeout, status), biến JSON thành DataFrame,
   và **lưu response thô** để tái lập.
3. Viết truy vấn SQL cơ bản (SELECT/WHERE/GROUP BY/JOIN) chạy bằng **DuckDB** —
   trực tiếp trên file CSV/Parquet.

In [ ]:
%pip install -q duckdb

import json, time
from pathlib import Path
import pandas as pd
import requests
import duckdb

Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)

## 1. Đọc file đúng cách

Bảng `listings` đầy đủ của Santiago có 90 cột và được nén dưới định dạng `.gz`.
Ví dụ này chỉ đọc các cột cần thiết; pandas có thể đọc trực tiếp file nén mà không cần giải nén trước.

In [ ]:
URL_FULL = ("https://data.insideairbnb.com/chile/rm/santiago/"
            "2026-06-29/data/listings.csv.gz")

df = pd.read_csv(
    URL_FULL,
    usecols=["id", "neighbourhood_cleansed", "room_type", "price",
             "host_since", "minimum_nights", "review_scores_rating"],
    parse_dates=["host_since"],
)
print(df.shape)
df.dtypes

`price` vẫn là **chuỗi** vì bảng đầy đủ lưu giá dạng `"$45,647.00"` (khác bản rút gọn ở buổi trước).
pandas không tự suy luận được quy ước tiền tệ này; có thể xử lý bằng `clean_price` từ các buổi trước.
ID Airbnb dài 18–19 chữ số cũng cần giữ ở kiểu số nguyên hoặc chuỗi để tránh mất độ chính xác.

In [ ]:
df["price_num"] = (df["price"]
                   .str.replace("$", "", regex=False)
                   .str.replace(",", "", regex=False)
                   .astype(float))
df[["price", "price_num"]].head(3)

### CSV vs Parquet — đo trên dữ liệu thật

In [ ]:
csv_path = Path("data/processed/listings.csv")
parquet_path = Path("data/processed/listings.parquet")
df.to_csv(csv_path, index=False)
df.to_parquet(parquet_path)

# Warm-up để thời gian không gồm chi phí nạp thư viện ở lần đọc đầu
pd.read_csv(csv_path)
pd.read_parquet(parquet_path)

for path, reader in [(csv_path, pd.read_csv), (parquet_path, pd.read_parquet)]:
    times = []
    for _ in range(5):
        t0 = time.perf_counter()
        reader(path)
        times.append(time.perf_counter() - t0)
    print(f"{path.name:22} {path.stat().st_size/1e6:5.1f} MB   đọc lại {min(times)*1000:5.0f} ms")

In [ ]:
# Parquet lưu dtype: host_since đọc lại vẫn là datetime, không phải chuỗi
pd.read_parquet("data/processed/listings.parquet").dtypes[["host_since", "price_num"]]

## 2. Gọi API thật: thời tiết Open-Meteo

[Open-Meteo](https://open-meteo.com) là API thời tiết miễn phí và không cần khóa API.
Ví dụ sau lấy dự báo nhiệt độ của Santiago trong 7 ngày tới:

In [ ]:
r = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={
        "latitude": -33.45, "longitude": -70.66,     # Santiago
        "daily": "temperature_2m_max,temperature_2m_min",
        "timezone": "auto",
    },
    timeout=20,          # giới hạn thời gian chờ
)
r.raise_for_status()      # phát sinh ngoại lệ nếu HTTP báo lỗi
print("Status:", r.status_code)

In [ ]:
d = r.json()              # từ điển lồng nhau
print(list(d.keys()))
print(json.dumps(d["daily"], ensure_ascii=False)[:150], "…")

In [ ]:
# Lưu response thô vào raw/ kèm ngày để hỗ trợ tái lập
tem = pd.Timestamp.now().strftime("%Y%m%d")
raw_path = Path(f"data/raw/openmeteo_santiago_{tem}.json")
raw_path.write_text(json.dumps(d, ensure_ascii=False))
print("Đã lưu:", raw_path)

# Từ đây, làm việc với file đã lưu thay vì gọi lại API
d2 = json.loads(raw_path.read_text())
thoi_tiet = pd.DataFrame(d2["daily"])
thoi_tiet

Ba biện pháp an toàn luôn cần thực hiện khi gọi API là đặt `timeout`, dùng `raise_for_status()` và
**lưu response thô trước khi xử lý**. Khi gọi API trong vòng lặp, cần thêm khoảng nghỉ
như `time.sleep(1)` giữa các yêu cầu.

## 3. SQL & DuckDB

DuckDB là cơ sở dữ liệu phân tích chạy **ngay trong notebook**, không cần máy chủ riêng
và có thể truy vấn **trực tiếp file CSV/Parquet**.

In [ ]:
duckdb.query("""
    SELECT room_type,
           median(price_num) AS gia_trung_vi,
           count(*)          AS n
    FROM 'data/processed/listings.parquet'
    GROUP BY room_type
    ORDER BY gia_trung_vi DESC
""").df()

In [ ]:
# Kiểm chứng chéo: pandas cần cho cùng kết quả
df.groupby("room_type")["price_num"].agg(["median", "size"])

Đối chiếu kết quả từ hai công cụ độc lập là một cách kiểm chứng hữu ích, kể cả khi mã được
tạo với sự hỗ trợ của AI. Tiếp theo, thực hiện JOIN với một DataFrame có sẵn trong notebook:

In [ ]:
vung_df = pd.DataFrame({
    "neighbourhood_cleansed": ["Santiago", "Providencia", "Las Condes", "Ñuñoa"],
    "vung": ["Trung tâm", "Đông", "Đông", "Đông"],
})

duckdb.query("""
    SELECT v.vung,
           median(l.price_num) AS gia,
           count(*)            AS n
    FROM 'data/processed/listings.parquet' AS l
    LEFT JOIN vung_df AS v USING (neighbourhood_cleansed)
    GROUP BY v.vung
    ORDER BY gia
""").df()

DuckDB có thể truy vấn trực tiếp `vung_df` của Python. Các giá trị `NULL`/`NaN`
là những quận chưa có trong bảng tra cứu, tương tự kết quả của phép left merge.

## 4. Bài tập tại lớp

### Bài 1 — read_csv có chủ đích

Bảng `reviews` **rút gọn** của Santiago chỉ có 2 cột (`listing_id`, `date`):
`https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/visualisations/reviews.csv`

Đọc nó với `parse_dates=["date"]`, rồi đếm số review theo **năm**
(gợi ý: dùng `df["date"].dt.year`; nội dung datetime sẽ được học kỹ ở buổi 8).
Năm nào nhiều review nhất?

In [ ]:
# TODO Bài 1:
rv = pd.read_csv(
    "https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/visualisations/reviews.csv",
    parse_dates=["date"],
)
rv["date"].dt.year.value_counts().sort_index().tail(5)

### Bài 2 — Gọi API cho thành phố khác

Rio de Janeiro nằm ở (`-22.91`, `-43.17`). Gọi Open-Meteo lấy nhiệt độ max 7 ngày tới của Rio,
rồi ghép với Santiago thành một bảng hai cột để so sánh (gợi ý: tạo hai DataFrame và
`merge` theo `time`). Trong bảy ngày dự báo, thành phố nào có nhiệt độ tối đa cao hơn?

In [ ]:
# TODO Bài 2:
r2 = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={"latitude": -22.91, "longitude": -43.17,
            "daily": "temperature_2m_max", "timezone": "auto"},
    timeout=20,
)
r2.raise_for_status()
rio = pd.DataFrame(r2.json()["daily"]).rename(columns={"temperature_2m_max": "rio_max"})
scl = thoi_tiet[["time", "temperature_2m_max"]].rename(columns={"temperature_2m_max": "santiago_max"})
so_sanh = scl.merge(rio, on="time")
so_sanh

### Bài 3 — Viết truy vấn SQL

Viết **một** truy vấn DuckDB trên `listings.parquet` trả về: các quận
(`neighbourhood_cleansed`) có **ít nhất 500 listing**, kèm giá trung vị và số listing,
xếp giảm dần theo giá trung vị. (Gợi ý: `HAVING count(*) >= 500`.)
Đối chiếu kết quả với chuỗi thao tác pandas tương đương ở buổi 5.

In [ ]:
# TODO Bài 3:
duckdb.query("""
    SELECT neighbourhood_cleansed,
           median(price_num) AS gia_trung_vi,
           count(*)          AS n
    FROM 'data/processed/listings.parquet'
    GROUP BY neighbourhood_cleansed
    HAVING count(*) >= 500
    ORDER BY gia_trung_vi DESC
""").df().head(5)

## 5. Bài tập về nhà — Xây dựng hàm thu thập snapshot

Viết script (hàm) `download_snapshot(city_path, date, dest)`:

1. Tải 3 file `listings.csv.gz`, `reviews.csv.gz`, `neighbourhoods.geojson` của một snapshot
   Inside Airbnb về `data/raw/<city>/<date>/` (dùng `requests`, ghi bytes; **bỏ qua nếu file
   đã tồn tại** để tránh tải lại).
2. Kiểm tra tối thiểu: file tồn tại, đọc được, số dòng > 0 — in báo cáo ngắn.
3. Chạy thử với thành phố nhóm bạn định chọn cho bài tập lớn.

Bài tập này thực hành bước thu thập và tổ chức dữ liệu gốc trong một pipeline.

In [ ]:
RUN_CHALLENGE = False

if RUN_CHALLENGE:
    def download_snapshot(city_path, date, dest="data/raw"):
        base = f"https://data.insideairbnb.com/{city_path}/{date}"
        ...
    download_snapshot("chile/rm/santiago", "2026-06-29")

---

## Tóm tắt buổi học

| Nội dung chính | Vì sao quan trọng |
|---|---|
| Giữ nguyên dữ liệu gốc; ghi kết quả xử lý vào `processed/` | Có thể truy vết và chạy lại pipeline |
| `usecols`/`parse_dates`/`na_values`; đọc file gz; Parquet lưu dtype | Kiểm soát cột, kiểu dữ liệu và bộ nhớ |
| API: params + timeout + raise_for_status + sleep + lưu response thô | Giảm lỗi kết nối và giữ đầu vào tái lập |
| DuckDB truy vấn trực tiếp CSV/Parquet và JOIN DataFrame | Không cần máy chủ riêng; có thể đối chiếu với pandas |

**Buổi sau:** xử lý dữ liệu chuỗi bằng `str.` và biểu thức chính quy (regex).